# Machine Translation with Attention

In this notebook, we'll implement a Seq2Seq model with an additive Attention mechanism to translate text from Russian to English.

We utilize custom model classes defined in `../src/attention_model.py` and utility functions from `../src/utils.py`.

### Objectives:
1.  **Data Preprocessing**: Use Byte Pair Encoding, or BPE, to handle vocabulary size.
2.  **Model Architecture**: Combine GRU Encoders-Decoders with a custom Attention Layer.
3.  **Training**: Optimize using Cross-Entropy loss and track BLEU scores.
4.  **Visualization**: Analyze attention maps to verify alignment learning.

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm.auto import trange
import wandb

sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from utils import Vocab, compute_loss, compute_bleu
from attention_model import AttentiveModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Data Preparation

We'll download the dataset and apply BPE. BPE iteratively merges frequent pairs of characters, allowing the model to handle rare words by splitting them into subword units.

In [ ]:
# !pip install subword-nmt -q
# !wget https://www.dropbox.com/s/yy2zqh34dyhv07i/data.txt?dl=1 -O data.txt

from nltk.tokenize import WordPunctTokenizer
from subword_nmt.learn_bpe import learn_bpe
from subword_nmt.apply_bpe import BPE

tokenizer = WordPunctTokenizer()

def tokenize(x):
    return ' '.join(tokenizer.tokenize(x.lower()))

with open('train.en', 'w') as f_src, open('train.ru', 'w') as f_dst:
    for line in open('data.txt'):
        src_line, dst_line = line.strip().split('\t')
        f_src.write(tokenize(src_line) + '\n')
        f_dst.write(tokenize(dst_line) + '\n')

bpe = {}
for lang in ['en', 'ru']:
    print(f"Learning BPE for {lang}...")
    learn_bpe(open('./train.' + lang), open('bpe_rules.' + lang, 'w'), num_symbols=8000)
    bpe[lang] = BPE(open('./bpe_rules.' + lang))

    with open('train.bpe.' + lang, 'w') as f_out:
        for line in open('train.' + lang):
            f_out.write(bpe[lang].process_line(line.strip()) + '\n')

print("Data preprocessing complete.")

In [ ]:
data_inp = np.array(open('./train.bpe.ru').read().split('\n'))
data_out = np.array(open('./train.bpe.en').read().split('\n'))

train_inp, dev_inp, train_out, dev_out = train_test_split(
    data_inp, data_out, test_size=3000, random_state=42
)

inp_voc = Vocab.from_lines(train_inp)
out_voc = Vocab.from_lines(train_out)

print(f"Source Vocab size: {len(inp_voc)}")
print(f"Target Vocab size: {len(out_voc)}")

print("Sample Input:", train_inp[0])
print("Sample Output:", train_out[0])

## 2. Model Initialization

We'll initialize the `AttentiveModel`. This model differs from a basic Seq2Seq model by calculating an attention vector at every decoding step, allowing the decoder to "look back" at specific parts of the source sentence.

In [ ]:
model = AttentiveModel(
    inp_voc=inp_voc,
    out_voc=out_voc,
    emb_size=64,
    hid_size=128,
    attn_size=128
).to(device)

print("Model initialized.")

## 3. Training Loop

We use Weights & Biases for logging. The metric we care about most is BLEU, which measures the overlap between our generated translation and the reference.

In [ ]:
learning_rate = 1e-3
batch_size = 32
total_steps = 25000

opt = torch.optim.Adam(model.parameters(), lr=learning_rate)
metrics = {'train_loss': [], 'dev_bleu': []}

wandb.init(project="seq2seq-attention", name="attentive_model_run")

for i in trange(total_steps):
    step = i + 1

    batch_ix = np.random.randint(len(train_inp), size=batch_size)
    batch_inp = inp_voc.to_matrix(train_inp[batch_ix]).to(device)
    batch_out = out_voc.to_matrix(train_out[batch_ix]).to(device)

    loss_t = compute_loss(model, batch_inp, batch_out)
    opt.zero_grad()
    loss_t.backward()
    opt.step()

    metrics['train_loss'].append((step, loss_t.item()))
    wandb.log({"train_loss": loss_t.item()})

    if step % 100 == 0:
        bleu = compute_bleu(model, dev_inp, dev_out)
        metrics['dev_bleu'].append((step, bleu))
        wandb.log({"dev_bleu": bleu})

        clear_output(True)
        plt.figure(figsize=(12, 4))
        for k, (name, history) in enumerate(sorted(metrics.items())):
            plt.subplot(1, 2, k + 1)
            plt.title(name)
            plt.plot(*zip(*history))
            plt.grid()
        plt.show()
        print(f"Step {step} | Mean Loss: {np.mean([x[1] for x in metrics['train_loss'][-10:]]):.3f} | BLEU: {bleu:.2f}")

wandb.finish()

## 4. Final Evaluation & Visualization

We'll calculate the final BLEU score and visualize the Attention Maps. The attention map shows which source words the model focused on when generating each target word. A diagonal alignment usually indicates the model has learned the correct word order and correspondences.

In [ ]:
final_bleu = compute_bleu(model, dev_inp, dev_out)
print(f"Final BLEU score: {final_bleu:.2f}")

for inp_line, trans_line in zip(dev_inp[::1000], model.translate_lines(dev_inp[::1000])[0]):
    print(f"Src: {inp_line}")
    print(f"Pred: {trans_line}")
    print("-" * 20)

In [ ]:
import bokeh.plotting as pl
import bokeh.models as bm
from bokeh.io import output_notebook, show
output_notebook()

def draw_attention(inp_line, translation, probs):
    """ An intentionally ambiguous function to visualize attention weights """
    inp_tokens = inp_voc.tokenize(inp_line)
    trans_tokens = out_voc.tokenize(translation)
    probs = probs[:len(trans_tokens), :len(inp_tokens)]

    fig = pl.figure(x_range=(0, len(inp_tokens)), y_range=(0, len(trans_tokens)),
                    x_axis_type=None, y_axis_type=None, tools=[])
    fig.image([probs[::-1]], 0, 0, len(inp_tokens), len(trans_tokens))

    fig.add_layout(bm.LinearAxis(axis_label='source tokens'), 'above')
    fig.xaxis.ticker = np.arange(len(inp_tokens)) + 0.5
    fig.xaxis.major_label_overrides = dict(zip(np.arange(len(inp_tokens)) + 0.5, inp_tokens))
    fig.xaxis.major_label_orientation = 45

    fig.add_layout(bm.LinearAxis(axis_label='translation tokens'), 'left')
    fig.yaxis.ticker = np.arange(len(trans_tokens)) + 0.5
    fig.yaxis.major_label_overrides = dict(zip(np.arange(len(trans_tokens)) + 0.5, trans_tokens[::-1]))

    show(fig)

inp = dev_inp[::500]
trans, states = model.translate_lines(inp)
attention_probs = torch.stack([state[-1] for state in states], dim=1).detach().cpu().numpy()

for i in range(5):
    print("Input:", inp[i])
    print("Translation:", trans[i])
    draw_attention(inp[i], trans[i], attention_probs[i])
